# Import libraries

In [1]:
import os
import re
from dotenv import load_dotenv
from itertools import groupby
from functools import partial
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_classic.indexes import SQLRecordManager, index
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

True

# Get original files

In this exercise, I'll use only the chapter 2 of the book

In [2]:
# Load all PDFs from data folder
data_dir = "data"
loader = PyPDFLoader(os.path.join(data_dir, "book_chapter_02.pdf"))
docs = loader.load()

print(f"Quantity of docs: {len(docs)}")
print("\nExample of doc:")
docs[0]

Quantity of docs: 34

Example of doc:


Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329110007', 'source': 'data\\book_chapter_02.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}, page_content='4 CHAPTER 2 • W ORDS AND TOKENS\nCHAPTER\n2\nWords and Tokens\nUser: I need some help, that much seems certain.\nELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP\nUser: Perhaps I could learn to get along with my mother.\nELIZA: TELL ME MORE ABOUT YOUR FAMILY\nUser: My mother takes care of me.\nELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU\nUser: My father.\nELIZA: YOUR FATHER\nUser: You are like my father in some ways.\nWeizenbaum (1966)\nThe dialogue above is fromELIZA, an early natural language processing systemELIZA\nthat could carry on a limited conversation with a user by imitating the responses of\na Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple\nprogram that uses pattern matching on words to recognize phrases like “I need X”\nand change the word

## Save only useful metadata 

In [3]:
keys_to_keep =["creationdate", "source", "total_pages", "page"]

for doc in docs:
    # Filter the dictionary
    doc.metadata = {k: v for k, v in doc.metadata.items() if k in keys_to_keep}

## Indexing

In [4]:
api_key = os.getenv("API_KEY")
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

In [5]:
get_vectorstore = partial(Chroma, embedding_function=embedding)

In [6]:
collection_name = "langchain_docs_index"

In [7]:
namespace = f"chroma/{collection_name}"
record_manager = SQLRecordManager(
    namespace=namespace,
    db_url="sqlite:///:memory:",
)
record_manager.create_schema()

### Parent Retriever

Por fines prácticos y económicos, trabajaré únicamente con el capítulo 2 del libro

La salida de la siguiente celda la pegué en Google AI Studio para saber la cantidad de tokens que representa. Con esta cantidad, puedo evaluar si está dentro del límite TPM de gemini-embedding-001

In [8]:
docs[0:(len(docs)//6)]

[Document(metadata={'creationdate': 'D:20260329110007', 'source': 'data\\book_chapter_02.pdf', 'total_pages': 34, 'page': 0}, page_content='4 CHAPTER 2 • W ORDS AND TOKENS\nCHAPTER\n2\nWords and Tokens\nUser: I need some help, that much seems certain.\nELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP\nUser: Perhaps I could learn to get along with my mother.\nELIZA: TELL ME MORE ABOUT YOUR FAMILY\nUser: My mother takes care of me.\nELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU\nUser: My father.\nELIZA: YOUR FATHER\nUser: You are like my father in some ways.\nWeizenbaum (1966)\nThe dialogue above is fromELIZA, an early natural language processing systemELIZA\nthat could carry on a limited conversation with a user by imitating the responses of\na Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple\nprogram that uses pattern matching on words to recognize phrases like “I need X”\nand change the words into suitable outputs like “What would it mean to you if yo

In [9]:
vectorstore = get_vectorstore(collection_name=f"{collection_name}_with_parents")
docstore = InMemoryStore()

In [10]:
# Split into smaller chunks
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # Average characters per chunk
    chunk_overlap=75,   # Overlap for context across chunks
    add_start_index=True # Add chunk index in metadata for traceability
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,     # Average characters per chunk
    chunk_overlap=50,   # Overlap for context across chunks
    add_start_index=True # Add chunk index in metadata for traceability
)

In [11]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [12]:
retriever.add_documents(docs[0:(len(docs)//6)])

In [ ]:
docs_response = retriever.invoke("Byte-Pair Encoding (BPE)")
docs_response

[Document(metadata={'creationdate': 'D:20260329110007', 'source': 'data\\book_chapter_02.pdf', 'total_pages': 34, 'page': 0, 'start_index': 0}, page_content='4 CHAPTER 2 • W ORDS AND TOKENS\nCHAPTER\n2\nWords and Tokens\nUser: I need some help, that much seems certain.\nELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP\nUser: Perhaps I could learn to get along with my mother.\nELIZA: TELL ME MORE ABOUT YOUR FAMILY\nUser: My mother takes care of me.\nELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU\nUser: My father.\nELIZA: YOUR FATHER\nUser: You are like my father in some ways.\nWeizenbaum (1966)\nThe dialogue above is fromELIZA, an early natural language processing systemELIZA\nthat could carry on a limited conversation with a user by imitating the responses of\na Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple\nprogram that uses pattern matching on words to recognize phrases like “I need X”\nand change the words into suitable outputs like “What would it

In [19]:
docs_response = vectorstore.similarity_search(
    "Byte-Pair Encoding (BPE)"
)

docs_response

[Document(id='fb243800-bcce-44a3-a225-90c385dc5e8e', metadata={'creationdate': 'D:20260329110007', 'source': 'data\\book_chapter_02.pdf', 'start_index': 2275, 'total_pages': 34, 'page': 0, 'doc_id': '47acc268-ab72-42d5-803a-1c061a6028b8'}, page_content='dard Byte-Pair Encoding (BPE) algorithm that automatically breaks up input textBPE\ninto tokens. This algorithm uses simple statistics of letter sequences to induce a\nvocabulary of subword tokens. All tokenization systems also depend on regular\nexpressions as a processing step. The regular expression is a language for formallyregular\nexpressions\nspecifying and manipulating text strings, an important tool in all modern NLP sys-'),
 Document(id='51bd9c84-39c7-4585-b493-10776c85077c', metadata={'doc_id': '47acc268-ab72-42d5-803a-1c061a6028b8', 'page': 0, 'start_index': 1799, 'creationdate': 'D:20260329110007', 'total_pages': 34, 'source': 'data\\book_chapter_02.pdf'}, page_content='Turkish, have very long words. We also need to think a